# كشكول التجارب: ربط واستخراج قواعد بيانات SQL

أهلاً بيك! قواعد بيانات SQL هي المخزن الأكبر لبيانات الشركات (جداول الموظفين، العملاء، الطلبات).  
عشان نستخدمها في الـ RAG، بنعمل استعلام SQL، وناخد كل صف ونحوله لنص مفهوم غني بالميتا-داتا.

في الكشكول ده هنجرب:
1. الاتصال بقاعدة بيانات `company.db` باستخدام `sqlite3`.
2. معرفة الجداول الموجودة واستخراج البيانات منها.
3. تحويل سجلات الموظفين لمستندات `Document` جاهزة للبحث الدلالي.


In [ ]:
# 1. الاتصال بقاعدة البيانات
from pathlib import Path

# نحدد مسار مجلد data تلقائياً
repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
data_dir = repo_root / "data"

import sqlite3
from langchain_core.documents import Document

db_file = data_dir / "databases" / "company.db"
conn = sqlite3.connect(str(db_file))
cursor = conn.cursor()

# نجيب أسماء الجداول
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]

print(f"الجداول الموجودة في قاعدة البيانات: {tables}")


### 2. استخراج بيانات الموظفين
تعال نعمل استعلام `SELECT` ونشوف البيانات اللي راجعة:


In [ ]:
cursor.execute("SELECT * FROM employees LIMIT 5;")
rows = cursor.fetchall()

# أسماء الأعمدة
cursor.execute("PRAGMA table_info(employees);")
cols = [col[1] for col in cursor.fetchall()]

print(f"أعمدة جدول الموظفين: {cols}\n")
for r in rows:
    print(r)


### 3. تحويل كل سجل لمستند RAG دلالي


In [ ]:
sql_docs = []

cursor.execute("SELECT * FROM employees;")
all_employees = cursor.fetchall()

for row in all_employees:
    row_dict = dict(zip(cols, row))
    content = f"الموظف {row_dict.get('name')} يشغل دور {row_dict.get('role')} في قسم {row_dict.get('department')} براتب {row_dict.get('salary')}."
    sql_docs.append(
        Document(
            page_content=content,
            metadata={"table": "employees", "emp_id": row_dict.get("id")}
        )
    )

conn.close()
print(f"تم تحويل {len(sql_docs)} سجل من SQL لمستندات جاهزة للـ RAG!")
print("\nعينة من أول مستند:")
print(sql_docs[0].page_content)


---
### مساحتك للتجربة الحرة
جرب تعمل استعلام SQL جديد، مثلاً هات الموظفين اللي راتبهم أعلى من رقم معين أو في قسم الـ Engineering!


In [ ]:
# جرب هنا:
# conn = sqlite3.connect(str(db_file))
# c = conn.cursor()
# c.execute("SELECT name, salary FROM employees WHERE department='Engineering'")
# print(c.fetchall())
# conn.close()
